In [4]:
import numpy as np
from scipy.optimize import fsolve

Ispin = 3/2
wkhz = 192.55*10**3
#Coefficient for LHQ (cluster 1) from ASICS
A = [-1.730215,-2.744504,-3.396561]
B = [1.251296,2.561384,-1.133846]
C = [3.573296,0.986618,-0.473004]
D = [-0.060956,-0.376258,-0.980912]
E = [-0.037878,-0.579924,-1.166911]

q = (3-4*Ispin*(Ispin + 1))/(16*wkhz)

# Function to solve the system of equations
# Function to solve the system of equations


def equations(vars, D, E, q):
    Axx, Ayy, Axy, Axz, Ayz = vars
    Azz = -(Axx + Ayy)  # Enforce the constraint directly

    # Equation system for Dx, Dy, Dz, Ex, Ey, Ez
    Dx = -((Ayy - Azz)**2 - 4*(Ayz)**2)*(9*q/8)
    Ex = Ayz*(Azz - Ayy)*(9*q/2)

    Dy = -((Axx - Azz)**2 - 4*(Axz)**2)*(9*q/8)
    Ey = Axz*(Azz - Axx)*(9*q/2)

    Dz = -((Axx - Ayy)**2 - 4*(Axy)**2)*(9*q/8)
    Ez = Axy*(Ayy - Axx)*(9*q/2)

    # Debugging prints
    # print(f"Axx: {Axx}, Ayy: {Ayy}, Azz: {Azz}, Axy: {Axy}, Axz: {Axz}, Ayz: {Ayz}")
    # print(f"Calculated Dx: {Dx}, Dy: {Dy}, Dz: {Dz}, Ex: {Ex}, Ey: {Ey}, Ez: {Ez}")


    # Equations to solve
    eq1 = Dx - D[0]
    eq2 = Ex - E[0]

    eq3 = Dy - D[1]
    eq4 = Ey - E[1]

    eq5 = Dz - D[2]
    eq6 = Ez - E[2]

    # Return only the first 5 equations as fsolve expects
    return [eq1, eq2, eq3, eq4, eq5]

#initial guess A_xx, A_yy, A_xy, A_xz, A_yz
initial_guess = [1, 1, 2, 2, 1]
# Solve the system
solution = fsolve(equations, initial_guess, args=(D, E, q))

# Extract the solutions
Axx, Ayy, Axy, Axz, Ayz = solution
Azz = -(Axx + Ayy)

# Print the results
print(f"Axx: {Axx}")
print(f"Ayy: {Ayy}")
print(f"Azz: {Azz}")
print(f"Axy: {Axy}")
print(f"Axz: {Axz}")
print(f"Ayz: {Ayz}")

# Ensure the calculated Ez matches the given E[2]
Ez = Axy * (Ayy - Axx) * (9 * q / 2)
print(f"Calculated Ez: {Ez}, Expected Ez: {E[2]}")


Axx: 114.68981459096969
Ayy: -39.784472951575246
Azz: -74.90534163939444
Axy: 248.85421929959102
Axz: -174.5072965562133
Ayz: -61.53065526052701
Calculated Ez: 0.6738007090126169, Expected Ez: -1.166911


In [ ]:
Dx = -((Ayy - Azz)**2 - 4*(Ayz)**2)*(9*q/8)
Ex = Ayz*(Azz - Ayy)*(9*q/2)

Dy = -((Axx - Azz)**2 - 4*(Axz)**2)*(9*q/8)
Ey = Axz*(Azz - Axx)*(9*q/2)

Dz = -((Axx - Ayy)**2 - 4*(Axy)**2)*(9*q/8)
Ez = Axy*(Ayy - Axx)*(9*q/2)

print(Dx, Dy, Dz, Ex, Ey, Ez)

In [ ]:
Q_T = np.zeros((3,3))
Q_T[0,0] = Axx; Q_T[0,1] = Axy; Q_T[0,2] = Axz
Q_T[1,0] = Axy; Q_T[1,1] = Ayy; Q_T[1,2] = Ayz;
Q_T[2,0] = Axz; Q_T[2,1] = Ayz; Q_T[2,2] = Azz;
print('Quadrupolar Tensor in Tenon frame: \n', Q_T)

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(Q_T)
D_quad = np.diag(eigenvalues)
print('Diagonalized Quadrupolar Tensor:\n', D_quad)
quad_avg = np.mean(eigenvalues)
sorted_eigenvalues = sorted((eigenvalues - quad_avg), key=abs)

Gzz_q = (sorted_eigenvalues[2] + quad_avg)*(2*Ispin*(2*Ispin - 1)) #following the Voseggard paper for principal frame parameters
Gxx_q = (sorted_eigenvalues[1] + quad_avg)*(2*Ispin*(2*Ispin - 1))
Gyy_q = (sorted_eigenvalues[0]+ quad_avg)*(2*Ispin*(2*Ispin - 1))

CQ_fit = Gzz_q/10**3
Qeta_fit = (Gyy_q - Gxx_q)/Gzz_q

print(CQ_fit, Qeta_fit)

In [ ]:
#Define symbol and force them to be real
AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy = sym.symbols('AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy', real=True)

#define set of equations
eq1 = sym.Eq(-((AzzmAyy)**2 - 4*(Ayz)**2)*((9*q/8)), D[0])
eq2 = sym.Eq(Ayz*(AzzmAyy)*(9*q/2), E[0])

eq3 = sym.Eq(-((AzzmAxx)**2 - 4*(Axz)**2)*(9*q/8), D[1])
eq4 = sym.Eq(Axz*(AzzmAxx)*(9*q/2), E[1])

eq5 = sym.Eq(-((AyymAxx)**2 - 4*(Axy)**2)*(9*q/8), D[2])
eq6 = sym.Eq(Axy*(AyymAxx)*(9*q/2), E[2])


# Solve the system
solution1 = sym.solve([eq1,eq2], (AzzmAyy,Ayz))

# Print the solutions
print(f"(Azz - Ayy) & Ayz: {solution1}")

solution2 = sym.solve([eq3,eq4], (AzzmAxx,Axz))

# Print the solutions
print(f"(Azz - Axx) & Axz: {solution2}")

solution3 = sym.solve([eq5,eq6], (AyymAxx,Axy))

# Print the solutions
print(f"(Ayy - Axx) & Axy: {solution3}")

AzzmAyy = [solution1[0][0], solution1[1][0]]
Ayz = [solution1[0][1], solution1[1][1]]
AzzmAxx = [solution2[0][0], solution2[1][0]]
Axz = [solution2[0][1], solution2[1][1]]
AyymAxx = [solution3[0][0], solution3[1][0]]
Axy = [solution3[0][1], solution3[1][1]]


# print(AzzmAyy, Ayz, AyymAxx)




In [2]:
import sympy as sym
from sympy import *
import itertools

In [8]:
w0 = 192.5 #larmor frequency (MHz)
Ispin = 3/2
wkhz = w0*10**3
#Coefficient for LHQ (cluster 1) from ASICS
A = [-1.730215,-2.744504,-3.396561]
B = [1.251296,2.561384,-1.133846]
C = [3.573296,0.986618,-0.473004]
D = [-0.060956,-0.376258,-0.980912]
E = [-0.037878,-0.579924,-1.166911]

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*wkhz)

In [9]:


#Function using fit parameters and q
def get_quad_combinations(fit_D,fit_E,q_value):

    #Define symbol and force them to be real
    AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy = sym.symbols('AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy', real=True)

    #Variable for each equation set
    quad_tensor = [(AzzmAyy, Ayz), (AzzmAxx, Axz), (AyymAxx, Axy)]

    # List to hold solutions
    solutions = []

    for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
        eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q_value/8)), fit_D[i])
        eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q_value/2), fit_E[i])

        # Solve the system
        solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
        solutions.append(solution)

        #    # Print the solutions
        # print(f"Set {i+1}: (Difference Variable & Off-diagonal): {solution}")

    # Assign the solutions to the respective variables
    AzzmAyy = [solutions[0][0][0], solutions[0][1][0]]
    Ayz = [solutions[0][0][1], solutions[0][1][1]]
    AzzmAxx = [solutions[1][0][0], solutions[1][1][0]]
    Axz = [solutions[1][0][1], solutions[1][1][1]]
    AyymAxx = [solutions[2][0][0], solutions[2][1][0]]
    Axy = [solutions[2][0][1], solutions[2][1][1]]

    return (AzzmAyy, Ayz, AzzmAxx, Axz, AyymAxx, Axy)


In [17]:
def get_best_quad_tensor(AzzmAxx, AyymAxx, AzzmAyy, Axz, Axy, Ayz):
    import numpy as np
    Axx1 = []; Axx2 = []; Axx3 = []
    Ayy1 = []; Ayy2 = []; Ayy3 = []
    Azz1 = []; Azz2 = []; Azz3 = []
    # Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
    combinations = list(itertools.product(AzzmAxx, AyymAxx, AzzmAyy))   
    # Initialize variables to track the best combination and minimum variation
    best_combination = None
    min_variation = float('inf')
    for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
        # Solution 1
        Axx1_val = -(AzzmAxx_val + AyymAxx_val)/3
        Ayy1_val = Axx1_val + AyymAxx_val
        Azz1_val = Axx1_val + AzzmAxx_val

        #Save values
        Axx1.append(Axx1_val)
        Ayy1.append(Ayy1_val)
        Azz1.append(Azz1_val)

        # Solution 2
        Ayy2_val = -(AzzmAyy_val - AyymAxx_val)/3
        Axx2_val = Ayy2_val - AyymAxx_val
        Azz2_val = Ayy2_val + AzzmAyy_val

        #Save values
        Axx2.append(Axx2_val)
        Ayy2.append(Ayy2_val)
        Azz2.append(Azz2_val)

        # Solution 3
        Azz3_val = (AzzmAxx_val + AzzmAyy_val)/3
        Axx3_val = Azz3_val - AzzmAxx_val
        Ayy3_val = Azz3_val - AzzmAyy_val
        
        #Save values
        Axx3.append(Axx3_val)
        Ayy3.append(Ayy3_val)
        Azz3.append(Azz3_val)

        # Convert sympy Float to regular Python float for NumPy functions
        Axx1_val = float(Axx1_val)
        Axx2_val = float(Axx2_val)
        Axx3_val = float(Axx3_val)
        
        Ayy1_val = float(Ayy1_val)
        Ayy2_val = float(Ayy2_val)
        Ayy3_val = float(Ayy3_val)
        
        Azz1_val = float(Azz1_val)
        Azz2_val = float(Azz2_val)
        Azz3_val = float(Azz3_val)

        # Calculate variation (standard deviation) for Axx, Ayy, Azz
        variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
        variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
        variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

        total_variation = variation_Axx + variation_Ayy + variation_Azz

        # Update the best combination if the current one has less variation
        if total_variation < min_variation:
            min_variation = total_variation
            best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
            best_Axx = np.mean([Axx1_val, Axx2_val, Axx3_val])
            best_Ayy = np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
            best_Azz = np.mean([Azz1_val, Azz2_val, Azz3_val])

    #Get index for off-diagonal elements        
    index_AzzmAxx = AzzmAxx.index(best_combination[0])
    best_Axz = Axz[index_AzzmAxx]

    index_AyymAxx = AyymAxx.index(best_combination[1])
    best_Axy = Axy[index_AyymAxx]

    index_AzzmAyy = AzzmAyy.index(best_combination[2])
    best_Ayz = Ayz[index_AzzmAyy]
       
    return best_combination, best_Axx, best_Ayy, best_Azz, best_Axz, best_Axy, best_Ayz
   


In [18]:
AzzmAyy, Ayz, AzzmAxx, Axz, AyymAxx, Axy = get_quad_combinations(D, E, q)
get_best_quad_tensor(AzzmAyy, Ayz, AzzmAxx, Axz, AyymAxx, Axy)

((-35.1163081394672, 61.5226657742180, -189.570538333426),
 -8.802119211583614,
 83.69773470254815,
 -74.89561549096453,
 -174.484637525737,
 248.999336095171,
 -267.298487496718)

In [5]:
#Define symbol for quadrupolar tensor and force them to be real
Azz_Q,Axx_Q,Ayy_Q,Ayz_Q,Axz_Q,Axy_Q = sym.symbols('Azz_Q,Axx_Q,Ayy_Q,Ayz_Q,Axz_Q,Axy_Q', real=True)

#Variable for each equation set
A = {
    'xx': Axx_Q, 'yy': Ayy_Q, 'zz': Azz_Q,
    'yz': Ayz_Q, 'zy': Ayz_Q,
    'xz': Axz_Q, 'zx': Axz_Q,
    'xy': Axy_Q, 'yx': Axy_Q,
}

#Define rotation tuple (a, b, g, bg, m)
rotations = [
    ('x', 'y', 'z', 'yz', 1),   # a = x, b = y, g = z, m = 1
    ('y', 'x', 'z', 'xz', 1),  # a = y, b = x, g = z, m = 1
    ('z', 'x', 'y', 'xy', -1)  # a = z, b = x, g = y, m = -1
]

# List to hold solutions for quadrupolar tensor terms
solutions_Q = []

for i in range(3):
    a, b, g, bg, m = rotations[i]
    eq1 = sym.Eq(
        -((A[b+b] - A[g+g])**2 - 4*(A[b+g])**2)*((9*q/8)), D[i]
        )
    eq2 = sym.Eq(
        A[b+g]*(A[g+g] - A[b+b])*(9*q/2), E[i]
        )
    eq3 = sym.Eq(
        A[a+a] + A[b+b] + A[g+g], 0
        )

    # Solve the system
    solution = sym.solve([eq1, eq2, eq3], (Azz_Q,Axx_Q,Ayy_Q,Ayz_Q,Axz_Q,Axy_Q))
    solutions_Q.append(solution)

print(solutions_Q)

# # Assign the solutions to the respective variables
# AzzmAyy_Q = [solutions_Q[0][0][0], solutions_Q[0][1][0]]
# Ayz_Q = [solutions_Q[0][0][1], solutions_Q[0][1][1]]
# AzzmAxx_Q = [solutions_Q[1][0][0], solutions_Q[1][1][0]]
# Axz_Q = [solutions_Q[1][0][1], solutions_Q[1][1][1]]
# AyymAxx_Q = [solutions_Q[2][0][0], solutions_Q[2][1][0]]
# Axy_Q = [solutions_Q[2][0][1], solutions_Q[2][1][1]]


# # Print the final results for the variables
# print(f"Azz - Ayy: {AzzmAyy_Q}, Ayz: {Ayz_Q}")
# print(f"Azz - Axx: {AzzmAxx_Q}, Axz: {Axz_Q}")
# print(f"Ayy - Axx: {AyymAxx_Q}, Axy: {Axy_Q}")  



[[(Azz_Q, 35.120868402864 - 2.0*Azz_Q, Azz_Q - 35.120868402864, 61.5306552120510, Axz_Q, Axy_Q), (Azz_Q, -2.0*Azz_Q - 35.120868402864, Azz_Q + 35.120868402864, -61.5306552120510, Axz_Q, Axy_Q)], [(Azz_Q, Azz_Q - 189.595156285395, 189.595156285395 - 2.0*Azz_Q, Ayz_Q, 174.507296397013, Axy_Q), (Azz_Q, Azz_Q + 189.595156285395, -2.0*Azz_Q - 189.595156285395, Ayz_Q, -174.507296397013, Axy_Q)], [(Azz_Q, 124.515835785858 - 0.5*Azz_Q, -0.5*Azz_Q - 124.515835785858, Ayz_Q, Axz_Q, -267.333199332134), (Azz_Q, -0.5*Azz_Q - 124.515835785858, 124.515835785858 - 0.5*Azz_Q, Ayz_Q, Axz_Q, 267.333199332134)]]
